In [ ]:
import os
import chromadb
from sentence_transformers import SentenceTransformer


BASE_DIR = os.path.dirname(__file__)
DOCS_DIR = os.path.join(BASE_DIR, "docs")
DB_DIR = os.path.join(BASE_DIR, "chroma_db")

COLLECTION_NAME = "zepto_policies"


documents = {
    "doc_01.txt": (
        "Zepto delivers grocery and household essentials to serviceable pin codes "
        "within 10 to 30 minutes of order confirmation, depending on the customer's "
        "delivery zone and current order volume. Standard delivery is free on orders over "
        "INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority "
        "delivery, which reserves the next available rider slot, is available at checkout "
        "for an additional INR 15. Zepto does not currently deliver to addresses outside "
        "its listed serviceable pin codes."
    ),

    "doc_02.txt": (
        "Grocery and perishable items may be reported for a return within 24 hours of delivery "
        "if damaged, spoiled, or incorrect; non-perishable packaged items may be returned "
        "within 7 days of delivery in unopened, resalable condition. Approved refunds are "
        "credited to the original payment method within 3–5 business days, or instantly to "
        "the Zepto wallet if the customer opts for wallet credit. Personal care items that "
        "have been opened are non-returnable except in the case of a manufacturing defect. "
        "Return pickup, where required, is arranged free of cost by Zepto."
    ),

    "doc_03.txt": (
        "Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), "
        "Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), "
        "and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early "
        "access to limited-time deals 24 hours before they go live to Basic and Pass members). "
        "Membership can be cancelled at any time from account settings; cancelling stops the next "
        "billing cycle but does not refund the current membership period."
    ),

    "doc_04.txt": (
        "Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, "
        "accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the "
        "rider moves. If an order's status shows no movement for more than 20 minutes past its original "
        "estimated delivery time, customers should contact support directly rather than continue waiting, "
        "since this indicates a likely delivery issue."
    ),

    "doc_05.txt": (
        "Orders can be cancelled free of cost any time before the order status changes to 'Packed', "
        "typically within the first 2 minutes of placing the order. Once an order has been packed, "
        "it can no longer be cancelled through the app, since the rider is dispatched immediately "
        "after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due "
        "to a Zepto-side issue (for example, rider unavailability), the order is auto-cancelled and "
        "fully refunded without any cancellation fee."
    ),

    "doc_06.txt": (
        "If an order arrives with damaged, spoiled, or missing items, customers must report it within "
        "24 hours of delivery through the 'Report an Issue' button on the order page. Zepto ships a "
        "free replacement or issues a full refund for damaged, spoiled, or missing items without "
        "requiring the customer to return the original item, unless the order value exceeds INR 1000, "
        "in which case a photo of the issue must be submitted through the report form before a "
        "replacement or refund is processed."
    ),

    "doc_07.txt": (
        "Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and "
        "INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid "
        "for 1 year from the date of issue and carry no maintenance fees. Gift card balance can be "
        "combined with one other payment method at checkout but cannot be combined with another gift "
        "card in the same transaction. Gift card balance cannot be redeemed for cash except where "
        "required by law."
    ),

    "doc_08.txt": (
        "Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given "
        "the time-sensitive nature of quick commerce deliveries. Average in-app chat response time "
        "is under 2 minutes. Email support is also available for non-urgent queries and is answered "
        "within 24 hours on business days. Phone support is not offered."
    )
}


# create the text files
os.makedirs(DOCS_DIR, exist_ok=True)

for name, text in documents.items():
    path = os.path.join(DOCS_DIR, name)

    with open(path, "w", encoding="utf-8") as file:
        file.write(text)

print("Policy files created.")


# load the embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")


# read the documents
texts = []
ids = []
metadata = []

for file_name in sorted(os.listdir(DOCS_DIR)):

    if file_name.endswith(".txt"):

        path = os.path.join(DOCS_DIR, file_name)

        with open(path, "r", encoding="utf-8") as file:
            text = file.read()

        texts.append(text)

        doc_id = file_name.replace(".txt", "")
        ids.append(doc_id)

        metadata.append({
            "source": file_name,
            "doc_id": doc_id
        })


# convert documents into embeddings
embeddings = model.encode(texts).tolist()

print("Embeddings created.")


# create ChromaDB
os.makedirs(DB_DIR, exist_ok=True)

client = chromadb.PersistentClient(path=DB_DIR)


# remove old collection if it exists
try:
    client.delete_collection(COLLECTION_NAME)
except:
    pass


collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)


# store everything in ChromaDB
collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings,
    metadatas=metadata
)

print("Documents added to ChromaDB.")
print("Total documents:", collection.count())


# test search
query = "What is the return window for perishable grocery items?"

query_embedding = model.encode([query]).tolist()

result = collection.query(
    query_embeddings=query_embedding,
    n_results=1
)

print("\nTest Query:")
print(query)

print("\nBest matching document:")
print(result["ids"][0][0])

print("\nDocument text:")
print(result["documents"][0][0])

In [2]:
import os
import chromadb
from sentence_transformers import SentenceTransformer

docs_folder = "docs"
db_folder = "chroma_db"


documents = {
    "doc_01.txt": (
        "Zepto delivers grocery and household essentials to serviceable pin codes "
        "within 10 to 30 minutes of order confirmation, depending on the customer's "
        "delivery zone and current order volume. Standard delivery is free on orders over "
        "INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority "
        "delivery, which reserves the next available rider slot, is available at checkout "
        "for an additional INR 15. Zepto does not currently deliver to addresses outside "
        "its listed serviceable pin codes."
    ),

    "doc_02.txt": (
        "Grocery and perishable items may be reported for a return within 24 hours of delivery "
        "if damaged, spoiled, or incorrect; non-perishable packaged items may be returned "
        "within 7 days of delivery in unopened, resalable condition. Approved refunds are "
        "credited to the original payment method within 3–5 business days, or instantly to "
        "the Zepto wallet if the customer opts for wallet credit. Personal care items that "
        "have been opened are non-returnable except in the case of a manufacturing defect. "
        "Return pickup, where required, is arranged free of cost by Zepto."
    ),

    "doc_03.txt": (
        "Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), "
        "Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), "
        "and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early "
        "access to limited-time deals 24 hours before they go live to Basic and Pass members). "
        "Membership can be cancelled at any time from account settings; cancelling stops the next "
        "billing cycle but does not refund the current membership period."
    ),

    "doc_04.txt": (
        "Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, "
        "accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the "
        "rider moves. If an order's status shows no movement for more than 20 minutes past its original "
        "estimated delivery time, customers should contact support directly rather than continue waiting, "
        "since this indicates a likely delivery issue."
    ),

    "doc_05.txt": (
        "Orders can be cancelled free of cost any time before the order status changes to 'Packed', "
        "typically within the first 2 minutes of placing the order. Once an order has been packed, "
        "it can no longer be cancelled through the app, since the rider is dispatched immediately "
        "after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due "
        "to a Zepto-side issue (for example, rider unavailability), the order is auto-cancelled and "
        "fully refunded without any cancellation fee."
    ),

    "doc_06.txt": (
        "If an order arrives with damaged, spoiled, or missing items, customers must report it within "
        "24 hours of delivery through the 'Report an Issue' button on the order page. Zepto ships a "
        "free replacement or issues a full refund for damaged, spoiled, or missing items without "
        "requiring the customer to return the original item, unless the order value exceeds INR 1000, "
        "in which case a photo of the issue must be submitted through the report form before a "
        "replacement or refund is processed."
    ),

    "doc_07.txt": (
        "Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and "
        "INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid "
        "for 1 year from the date of issue and carry no maintenance fees. Gift card balance can be "
        "combined with one other payment method at checkout but cannot be combined with another gift "
        "card in the same transaction. Gift card balance cannot be redeemed for cash except where "
        "required by law."
    ),

    "doc_08.txt": (
        "Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given "
        "the time-sensitive nature of quick commerce deliveries. Average in-app chat response time "
        "is under 2 minutes. Email support is also available for non-urgent queries and is answered "
        "within 24 hours on business days. Phone support is not offered."
    )
}


os.makedirs(docs_folder, exist_ok=True)

for file_name, text in documents.items():
    file_path = os.path.join(docs_folder, file_name)

    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text)


print("Policy documents created.")


model = SentenceTransformer("all-MiniLM-L6-v2")


texts = []
ids = []
metadatas = []


for file_name in sorted(os.listdir(docs_folder)):
    if file_name.endswith(".txt"):

        file_path = os.path.join(docs_folder, file_name)

        with open(file_path, "r", encoding="utf-8") as file:
            text = file.read().strip()

        texts.append(text)
        ids.append(file_name[:-4])
        metadatas.append({"source": file_name})


print("Documents loaded:", len(texts))


embeddings = model.encode(texts).tolist()

print("Embeddings created.")


os.makedirs(db_folder, exist_ok=True)

client = chromadb.PersistentClient(path=db_folder)

collection = client.get_or_create_collection(
    name="zepto_policies",
    metadata={"hnsw:space": "cosine"}
)


collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings,
    metadatas=metadatas
)


print("Documents stored in ChromaDB:", collection.count())

Policy documents created.


c:\Users\Putti Sampath\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Putti Sampath\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 566

Documents loaded: 8
Embeddings created.
Documents stored in ChromaDB: 8


In [3]:
prompt = """
You are a Zepto customer support assistant.
Use the information given in the context to answer the customer's question.
Context:{context}
Customer question:{question}
Give a clear and short answer based on the context.
Keep the answer within 2 to 4 sentences.
Do not use information that is not present in the given context.
If the context does not contain the answer, say that the information is not available.
Example:
Context:
Zepto gift cards are valid for 1 year from the date of issue.

Question:
How long is a Zepto gift card valid?

Answer:
A Zepto gift card is valid for 1 year from the date of issue.

Now answer the customer's question using the context above.
"""

Task 3

In [ ]:
import os
from typing import TypedDict

from langgraph.graph import StateGraph, START, END

mock_LLM = os.getenv("MOCK_LLM", "1")

class SupportState(TypedDict):
    question: str
    intent: str
    answer: str


def classify_intent(state: SupportState):
    question = state["question"].lower()
    keywords = [
        "delivery",
        "return",
        "refund",
        "membership",
        "tracking",
        "cancel",
        "gift card",
        "support hours"
    ]

    if any(word in question for word in keywords):
        intent = "policy_question"
    else:
        intent = "general_question"
    return {"intent": intent}


def retrieve_and_answer(state: SupportState):
    question = state["question"]
    query_embedding = model.encode(question).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )
    retrieved_docs = results["documents"][0]

    if mock_LLM != "0":
        top_chunk = retrieved_docs[0]
        answer = f"Based on the retrieved context: {top_chunk[:200]}"
        return {"answer": answer}

    context = "\n\n".join(retrieved_docs)

    prompt = prompt.format(
        context=context,
        question=question
    )
    answer = "Real LLM answer goes here."
    return {"answer": answer}


def direct_answer(state: SupportState):
    question = state["question"]
    if mock_LLM != "0":
        return {
            "answer": "I can only answer questions about Zepto policies right now."
        }
    prompt = prompt.format(
        context="",
        question=question
    )
    answer = "Real LLM answer goes here."
    return {"answer": answer}


def route_question(state: SupportState):
    if state["intent"] == "policy_question":
        return "retrieve_and_answer"
    return "direct_answer"


graph = StateGraph(SupportState)
graph.add_node("classify_intent", classify_intent)
graph.add_node("retrieve_and_answer", retrieve_and_answer)
graph.add_node("direct_answer", direct_answer)
graph.add_edge(START, "classify_intent")
graph.add_conditional_edges(
    "classify_intent",
    route_question,
    {
        "retrieve_and_answer": "retrieve_and_answer",
        "direct_answer": "direct_answer"
    }
)
graph.add_edge("retrieve_and_answer", END)
graph.add_edge("direct_answer", END)
app = graph.compile()

In [5]:
result = app.invoke({
    "question": "How much is the delivery fee?",
    "intent": "",
    "answer": ""
})
print(result["answer"])

Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del


In [6]:
result = app.invoke({
    "question": "What is the capital of India?",
    "intent": "",
    "answer": ""
})
print(result["answer"])

I can only answer questions about Zepto policies right now.


In [7]:
from pydantic import BaseModel, Field

In [8]:
class SupportResponse(BaseModel):
    answer: str
    sources: list[str]
    confidence: float = Field(ge=0.0, le=1.0)

In [9]:
class SupportState(TypedDict):
    question: str
    intent: str
    answer: dict

In [ ]:
def retrieve_and_answer(state: SupportState):
    question = state["question"]

    query_embedding = model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    retrieved_docs = results["documents"][0]
    retrieved_ids = results["ids"][0]

    if mock_LLM != "0":
        top_chunk = retrieved_docs[0]
        response = SupportResponse(
            answer=f"Based on the retrieved context: {top_chunk[:200]}",
            sources=retrieved_ids,
            confidence=1.0
        )
        return {"answer": response.model_dump()}
    context = "\n\n".join(retrieved_docs)

    prompt = prompt.format(
        context=context,
        question=question
    )
    answer = "Real LLM answer goes here."

    response = SupportResponse(
        answer=answer,
        sources=retrieved_ids,
        confidence=1.0
    )
    return {"answer": response.model_dump()}

In [ ]:
def direct_answer(state: SupportState):
    if mock_LLM != "0":
        response = SupportResponse(
            answer="I can only answer questions about Zepto policies right now.",
            sources=[],
            confidence=1.0
        )
        return {"answer": response.model_dump()}
    prompt = prompt.format(
        context="",
        question=state["question"]
    )
    answer = "Real LLM answer goes here."

    response = SupportResponse(
        answer=answer,
        sources=[],
        confidence=1.0
    )
    return {"answer": response.model_dump()}

In [12]:
result = app.invoke({
    "question": "How much is the delivery fee?",
    "intent": "",
    "answer": {}
})
print(result["answer"])

Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del


In [13]:
result = app.invoke({
    "question": "What is the capital of India?",
    "intent": "",
    "answer": {}
})
print(result["answer"])

I can only answer questions about Zepto policies right now.


Task 5

In [14]:
from fastapi import FastAPI
from pydantic import BaseModel

In [15]:
import os
from typing import TypedDict

import chromadb
from sentence_transformers import SentenceTransformer
from langgraph.graph import StateGraph, START, END
from fastapi import FastAPI
from pydantic import BaseModel, Field

In [17]:
class AskRequest(BaseModel):
    query: str
    
{
    "query": "How much is the delivery fee?"
}

api = FastAPI()

@api.post("/ask", response_model=SupportResponse)
def ask_question(request: AskRequest):

    result = app.invoke({
        "question": request.query,
        "intent": "",
        "answer": {}
    })

    return result["answer"]